# Module 4: Advanced Data Wrangling & Transformation
## Objective: Mastering high-level transformations and professional code structures.

In this module, we will learn how to inject custom logic into our dataframes, 
calculate moving averages for trend analysis, and reshape our tables for 
advanced visualization.

### 1. Row-Wise and Element-Wise Operations
* Use `.map()` when you have a simple dictionary or a single-input function for one column.
* Use `.apply()` when your logic needs to look at multiple columns at once.

**Scenario**: Categorizing regional risk based on both population density and daily growth.

In [1]:
import pandas as pd
import numpy as np

# Sample Data
df = pd.DataFrame({
    'City': ['Peshawar', 'Mardan', 'Swat'],
    'Growth_Rate': [0.05, 0.02, 0.08],
    'Status_Code': [1, 2, 1]
})

# .map(): Mapping codes to readable labels
status_map = {1: 'Active', 2: 'Stable'}
df['Status_Label'] = df['Status_Code'].map(status_map)

# .apply(): Complex logic involving multiple columns
def calculate_risk(row):
    if row['Growth_Rate'] > 0.06 and row['Status_Label'] == 'Active':
        return 'High'
    return 'Low'

df['Risk_Level'] = df.apply(calculate_risk, axis=1)
display(df)

,City,Growth_Rate,Status_Code,Status_Label,Risk_Level
0,Peshawar,0.05,1,Active,Low
1,Mardan,0.02,2,Stable,Low
2,Swat,0.08,1,Active,High


### 2. Rolling Statistics
Window functions allow us to perform calculations on a "sliding" window of data. 
This is the standard way to calculate **Moving Averages**.

In [2]:
# Create 30 days of synthetic sales data
dates = pd.date_range(start='2024-01-01', periods=30)
sales = np.random.randint(100, 500, size=30)
ts_df = pd.DataFrame({'Date': dates, 'Sales': sales})

# Calculate 7-Day Rolling Average
ts_df['7Day_Avg'] = ts_df['Sales'].rolling(window=7).mean()

print("First 10 days of Sales and Rolling Average:")
display(ts_df.head(10))

First 10 days of Sales and Rolling Average:


,Date,Sales,7Day_Avg
0,2024-01-01,440,NaN
1,2024-01-02,204,NaN
2,2024-01-03,489,NaN
3,2024-01-04,146,NaN
4,2024-01-05,357,NaN
5,2024-01-06,273,NaN
6,2024-01-07,483,341.714286
7,2024-01-08,364,330.857143
8,2024-01-09,347,351.285714
9,2024-01-10,325,327.857143


### 3. Reshaping: From Wide to Long
Most modern visualization libraries (like Seaborn) require data in **Long Format**. 
However, data often arrives in **Wide Format** from Excel.

In [3]:
# Wide Format (Excel style)
wide_df = pd.DataFrame({
    'City': ['Peshawar', 'Mardan'],
    'Jan_Sales': [500, 300],
    'Feb_Sales': [600, 400]
})

# Melting: Wide -> Long
long_df = pd.melt(wide_df, id_vars=['City'], var_name='Month', value_name='Revenue')

print("Long Format (Optimized for Analysis):")
display(long_df)

# Pivoting: Long -> Wide (Back to Summary style)
summary_df = long_df.pivot(index='City', columns='Month', values='Revenue')
print("\nPivot Table Summary:")
display(summary_df)

Long Format (Optimized for Analysis):


,City,Month,Revenue
0,Peshawar,Jan_Sales,500
1,Mardan,Jan_Sales,300
2,Peshawar,Feb_Sales,600
3,Mardan,Feb_Sales,400



Pivot Table Summary:


Month,Feb_Sales,Jan_Sales
City,,
Mardan,400,300
Peshawar,600,500


### 4. Professional Method Chaining
Instead of writing 5 separate lines of code, we wrap our operations in 
parentheses `()` to create a clean, readable pipeline.

In [4]:
# Professional pipeline
cleaned_df = (
    pd.DataFrame(long_df)
    .rename(columns={'Revenue': 'Monthly_Rev'})
    .assign(Tax = lambda x: x['Monthly_Rev'] * 0.15)
    .query('Monthly_Rev > 350')
    .sort_values('Monthly_Rev', ascending=False)
)

print("Result of Chained Pipeline:")
display(cleaned_df)

Result of Chained Pipeline:


,City,Month,Monthly_Rev,Tax
2,Peshawar,Feb_Sales,600,90.0
0,Peshawar,Jan_Sales,500,75.0
3,Mardan,Feb_Sales,400,60.0


### Student Exercise: The Regional Analyst Pipeline
**Goal**: Process a messy dataset using advanced transformations.
1. Create a "Wide" dataframe with 4 cities and their monthly rainfall for 3 months.
2. **Melt** the data into "Long" format.
3. Use **.apply()** to create a new column called 'Intensity' (If rain > 100 'High', else 'Low').
4. Use **Method Chaining** to:
    - Filter for 'High' intensity only.
    - Sort by Rainfall.
    - Calculate a 15% 'Runoff' column using `.assign()`.